In [88]:
import tensorflow as tf
import time
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [89]:
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever",
    "I dislike this movie",
    "The film was awful"
]

In [90]:
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

In [91]:
# Token Embeddings
# Converting the text data into sequences and then padding them to maintain consistent input length for the model

vocab_size = 1000
max_length = 6


tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_length, padding='post')

print("Word Index:")
print(tokenizer.word_index)


print("\nInput Sequences:")
print(X)

Word Index:
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'ever': 18, 'dislike': 19, 'the': 20, 'was': 21, 'awful': 22}

Input Sequences:
[[ 3  9  2  4  0  0]
 [ 2  5 10 11  0  0]
 [ 6 12  7  0  0  0]
 [13  8  0  0  0  0]
 [ 3 14  2  4  0  0]
 [15  5  0  0  0  0]
 [ 6 16  8  0  0  0]
 [17  7 18  0  0  0]
 [ 3 19  2  4  0  0]
 [20  5 21 22  0  0]]


In [92]:
#Positional Embeddings
# Model tries to learn the position of each word in the sequence, to understand the context. And then it translates the input data.
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        # Token Embedding
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
       # Positional Embedding
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [93]:
#Multi-Head Attention and Feed-Forward Networks
# The attention allows the model to focus on different parts of the input sequence when making predictions, feed forward network helps in learning patterns and relationships in the data.

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        # Multi-Head Attention
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
       # Feed Forward Network
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # Attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        # The layer normalization 
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        # The output from the previous step is passed through a feed forward network to the next layer.
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [94]:
# build the model
embed_dim = 32
num_heads = 8
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
# Output layer
# The output from the transformer block is passed through a global average pooling layer to reduce the dimensionality.
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [95]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
    )

model.summary() 

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_8  │ (None, 6, 32)          │        32,192 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_8             │ (None, 6, 32)          │        35,808 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,033 (265.75 KB)

 Trainable params: 68,033 (265.75 KB)

 Non-trainable params: 0 (0.00 B)

In [96]:
model.fit(X, labels, epochs=30, batch_size=2,verbose=1)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.3000 - loss: 0.7890
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5000 - loss: 0.7695 
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6000 - loss: 0.6194 
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5000 - loss: 0.6179
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6000 - loss: 0.6107 
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6000 - loss: 0.5909     
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7000 - loss: 0.4863 
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8000 - loss: 0.4415 
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.4350 
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.3605 
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9000 - loss: 0.3324 
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9000 - loss: 0.285

In [97]:
test_sentences = [
    "I hate the movie",
    "This story is excellent and the acting is terrible",
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
I hate the movie -> 0.0018535418
Prediction: Negative
This story is excellent and the acting is terrible -> 0.38486677
Prediction: Negative


# Task 2 Data Flow Diagram

```mermaid
flowchart LR
    A[Input text sequences] --> B[Tokenizer + Padding]
    B --> C[Token Embedding]
    C --> D[Positional Embedding]
    D --> E[Add token + position vectors]
    E --> F[Multi-Head Attention]
    F --> G[Add & Normalize]
    G --> H[Feed Forward Network]
    H --> I[Add & Normalize]
    I --> J[GlobalAveragePooling1D]
    J --> K[Dense Output Layer]
    K --> L[Sigmoid prediction]
```


## Task 3
1. Token Embedding
 We created a token embedding layer that converts each word index in the input sequence into a dense vector. This is what allows the model to work with meaningful numeric representations instead of raw token IDs.
2. Positional Embedding
 We added a positional embedding layer so the model can learn the order of words in the sentence. This is necessary because transformers do not naturally know sequence position on their own.
3. Multi-Head Attention
 We used multi-head attention inside the transformer block to let the model focus on different words in the sentence at the same time. This helps it capture relationships and context between tokens more effectively.
4. Feed Forward Network
 We included a small feed-forward neural network after attention to further transform the features learned from the sequence. This adds extra non-linearity and helps the model learn more complex patterns.
5. Add & Normalize
 We used residual connections with layer normalization after both the attention layer and the feed-forward network. This helps keep training stable and makes it easier for gradients and information to flow through the model.
6. Output Layer
 We ended the model with a sigmoid dense output layer that produces a probability for binary classification. In this code, it gives the final sentiment prediction as positive or negative.